# Agent Orchestration: The Execution Substrate

**Level:** Advanced · **Time:** 60 min

In the Agent Coordination module, we discussed how agents talk to *each other*. In this module, we discuss the **Orchestrator**: the deterministic software engine that decides *what runs next*, *how state is saved*, and *when to sleep for human approval*.

We will explore three different paradigms using the **Northstar EU Checkout Incident**:
1. **DAGs (Pipelines)** for parallel evidence gathering.
2. **State Machines (Graphs)** for cyclic agent reasoning.
3. **Durable Execution** for robust, multi-day, crash-proof execution.

> **Note:** The code blocks in this notebook are written to be fully syntactically correct against their respective SDKs (LangGraph, Temporal). To allow this notebook to run without requiring a live Temporal server or LLM API keys locally, the outputs are simulated as they would appear in production.

---
## Pattern 1: Directed Acyclic Graphs (DAGs)

![DAG Orchestration](../../../assets/orch_dag.svg)

A DAG executes dependency-ready work. It is deterministic, strictly flows in one direction (no loops), and is ideal for parallelizing independent operations to reduce wall-clock time.

**Framework of Choice:** Apache Airflow, Dagster, or standard Python `asyncio`.

Here, we use `asyncio.gather` to represent a DAG that fetches Metrics, Logs, and Deployments simultaneously.

In [ ]:
import asyncio
import time

async def fetch_metrics():
    print("[DAG] Fetching Datadog metrics...")
    await asyncio.sleep(1)
    return "504 Spike Detected"

async def fetch_logs():
    print("[DAG] Fetching Kubernetes logs...")
    await asyncio.sleep(1)
    return "Error: Upstream Timeout"

async def fetch_deployments():
    print("[DAG] Fetching GitHub Deployments...")
    await asyncio.sleep(1)
    return "deploy-842 rolled out"

async def run_dag():
    start_time = time.time()
    
    # The DAG branches out in parallel
    results = await asyncio.gather(
        fetch_metrics(),
        fetch_logs(),
        fetch_deployments()
    )
    
    # The DAG joins
    print(f"\n[DAG] Joined Results: {results}")
    print(f"[DAG] Completed in {time.time() - start_time:.2f} seconds")

# await run_dag()

[DAG] Fetching Datadog metrics...
[DAG] Fetching Kubernetes logs...
[DAG] Fetching GitHub Deployments...

[DAG] Joined Results: ['504 Spike Detected', 'Error: Upstream Timeout', 'deploy-842 rolled out']
[DAG] Completed in 1.01 seconds


**Selection Criteria:** Use a DAG when you have rigid, parallelizable tasks that do not require an LLM to dynamically determine the next route. Do *not* use a DAG if you need a "retry loop" where an agent checks its own work.

---
## Pattern 2: State Machines (Graphs)

![State Machine Orchestration](../../../assets/orch_state_machine.svg)

A State Machine explicitly defines states and transitions. Unlike a DAG, it supports **cycles/loops**. This is essential for Agent reasoning, where an Agent might invoke a tool, review the result, and decide to try again.

**Framework of Choice:** [LangGraph](https://langchain-ai.github.io/langgraph/).

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    evidence: str
    proposed_action: str
    confidence: float

def agent_node(state: AgentState):
    print("[State Machine] Agent analyzing evidence...")
    # Simulate an LLM producing a proposal with low confidence first, then high
    if state.get("confidence", 0) < 0.9:
        return {"proposed_action": "Need more data", "confidence": 0.95}
    return {"proposed_action": "Rollback deploy-842", "confidence": 0.99}

def review_node(state: AgentState):
    print(f"[State Machine] Reviewing proposed action: {state['proposed_action']}")
    if state["confidence"] < 0.9:
        return "reject"
    return "approve"

workflow = StateGraph(AgentState)
workflow.add_node("Agent", agent_node)

# Conditional Routing allows CYCLES
workflow.add_conditional_edges("Agent", review_node, {
    "reject": "Agent",  # Cycle back to the agent
    "approve": END      # Terminal state
})

workflow.set_entry_point("Agent")
app = workflow.compile()

# result = app.invoke({"evidence": "504 Spike, deploy-842"})

[State Machine] Agent analyzing evidence...
[State Machine] Reviewing proposed action: Need more data
[State Machine] Agent analyzing evidence...
[State Machine] Reviewing proposed action: Rollback deploy-842
Final State: {'proposed_action': 'Rollback deploy-842', 'confidence': 0.99}


**Selection Criteria:** Use State Machines for dynamic Agent reasoning and loops. Do *not* use a raw State Machine for long-running workflows (days) unless backed by a robust persistence and checkpointer layer.

---
## Pattern 3: Durable Execution (Checkpoints & Persistence)

![Durable Execution](../../../assets/orch_durable.svg)

Durable execution guarantees that if your process crashes (e.g., OOM kill, server restart), it will resume exactly where it left off. It treats a Python function as a recoverable workflow.

**Framework of Choice:** [Temporal](https://temporal.io/).

In [ ]:
from temporalio import workflow, activity
from datetime import timedelta

@activity.defn
async def apply_rollback(action: str) -> str:
    print(f"[Durable Execution] Actioning: {action}")
    return "Rollback Successful"

@workflow.defn
class IncidentWorkflow:
    @workflow.run
    async def run(self, action_fingerprint: str) -> str:
        
        # 1. Checkpoint and Sleep
        print("[Durable Execution] Checkpointing state to database.")
        print("[Durable Execution] Sleeping. CPU is freed. Awaiting Human Approval Event...")
        
        # The workflow sleeps safely. If the server crashes here, 
        # Temporal will wake it up perfectly on another server when the signal arrives.
        human_approved = await workflow.wait_condition(
            lambda: self.is_approved, 
            timeout=timedelta(hours=24)
        )
        
        if not human_approved:
            return "Escalated: Timeout"
            
        # 2. Re-validate and Execute Activity
        print("[Durable Execution] Event Received. Re-validating identity...")
        result = await workflow.execute_activity(
            apply_rollback, 
            action_fingerprint, 
            schedule_to_close_timeout=timedelta(minutes=5)
        )
        return result

    @workflow.signal
    def approve(self) -> None:
        self.is_approved = True

# Client execution
# client = await Client.connect("localhost:7233")
# handle = await client.start_workflow(IncidentWorkflow.run, "proposal:rollback:checkout:deploy-842")
# await handle.signal(IncidentWorkflow.approve)

[Durable Execution] Checkpointing state to database.
[Durable Execution] Sleeping. CPU is freed. Awaiting Human Approval Event...
[Temporal Server] Received Approval Signal. Waking Workflow.
[Durable Execution] Event Received. Re-validating identity...
[Durable Execution] Actioning: proposal:rollback:checkout:deploy-842
Workflow Result: Rollback Successful


---
## Conclusion

In production architectures, **these patterns are rarely mutually exclusive**. 
A state-of-the-art enterprise system often uses:
1. **Temporal (Durable Execution)** as the outer shell to guarantee the incident ticket survives for days and can wait for human approval.
2. **LangGraph (State Machine)** inside a Temporal Activity to allow an agent to loop and synthesize data.
3. **DAGs (Pipelines)** to rapidly fetch data dependencies in parallel before handing them to the LangGraph node.